# M5 non-recursive LightGBM: one store

A step-by-step, single-store version of the `2-1. nonrecursive_store_TRAIN` script from the M5 winning solution.

**Steps**
1. Set the options in the config cell.
2. Load one store's features from the preprocessing pickles.
3. Split by day (default `cv4`: train up to day 1885, validate on days 1886 to 1913).
4. Compute two baselines (all zeros, and same weekday last week).
5. Train LightGBM (Tweedie objective) with early stopping.
6. Compare the model with the baselines, then save the model, predictions and run details.

This is **not** the official WRMSSE metric (that needs all 12 hierarchy levels). Compare the model with the baselines printed here, not with leaderboard numbers.

**Needs:** `lightgbm`, `pandas`, `numpy`, and the pickles `grid_part_1/2/3.pkl`, `lags_df_28.pkl`, `mean_encoding_df.pkl` from the preprocessing step.

## 1. Config (edit this cell)

In [ ]:
from pathlib import Path

DATA_DIR = Path("D:/m5/processed")   # folder that holds the .pkl files from preprocessing
OUT_DIR = Path("outputs")            # where the model, predictions and run details are saved

STORE = "CA_1"          # store id, e.g. CA_1, TX_2, WI_3
CV = "cv4"              # cv1, cv2, cv3, cv4 or public (not "private": it has no labels)
FIRST_DAY = 710         # ignore days before this (the original script uses 710)
PRESET = "dev"          # "dev" = faster, "original" = the author's slow 3000-tree setting
EARLY_STOPPING = 50     # stop after this many rounds without improvement (0 = off)
MAX_ITEMS = None        # e.g. 500 to use only the first 500 series for a quick test
NUM_THREADS = 0         # 0 = LightGBM default
FORCE = False           # True = retrain even if a saved model exists

## 2. Imports and constants

In [ ]:
from __future__ import annotations

import gc
import json
import logging
import time
from datetime import datetime, timezone
from typing import Any

import lightgbm as lgb
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
LOG = logging.getLogger("m5_train")

TARGET = "sales"

# (train_end_day, valid_end_day): validate on days in (train_end, valid_end].
# "private" (d=1942..1969) is left out on purpose: those days have no labels.
FOLDS: dict[str, tuple[int, int]] = {
    "cv1": (1551, 1610),
    "cv2": (1829, 1857),
    "cv3": (1857, 1885),
    "cv4": (1885, 1913),
    "public": (1913, 1941),
}

REMOVE_FEATURES = ["id", "state_id", "store_id", "date", "wm_yr_wk", "d", TARGET]

GRID2_COLS = [
    "sell_price", "price_max", "price_min", "price_std", "price_mean",
    "price_norm", "price_nunique", "item_nunique",
    "price_momentum", "price_momentum_m", "price_momentum_y",
]
GRID3_COLS = [
    "event_name_1", "event_type_1", "event_name_2", "event_type_2",
    "snap_CA", "snap_TX", "snap_WI",
    "tm_d", "tm_w", "tm_m", "tm_y", "tm_wm", "tm_dw", "tm_w_end",
]
LAG_COLS = (
    [f"sales_lag_{k}" for k in range(28, 43)]
    + [f"rolling_{stat}_{w}" for w in (7, 14, 30, 60, 180) for stat in ("mean", "std")]
)
MEAN_ENC_COLS = [
    "enc_store_id_dept_id_mean", "enc_store_id_dept_id_std",
    "enc_item_id_state_id_mean", "enc_item_id_state_id_std",
]

BASE_PARAMS: dict[str, Any] = {
    "boosting_type": "gbdt",
    "objective": "tweedie",
    "tweedie_variance_power": 1.1,
    "metric": "rmse",
    "subsample": 0.5,
    "subsample_freq": 1,
    "feature_fraction": 0.5,
    "max_bin": 100,
    "boost_from_average": False,
    "verbose": -1,
    "seed": 1995,
}

# "dev" is a faster setting for experiments. "original" matches the author's
# settings (3000 trees at lr 0.015) and is much slower.
PRESETS: dict[str, dict[str, Any]] = {
    "dev": {"learning_rate": 0.05, "num_leaves": 255, "min_data_in_leaf": 255, "rounds": 1500},
    "original": {"learning_rate": 0.015, "num_leaves": 255, "min_data_in_leaf": 255, "rounds": 3000},
}

assert CV in FOLDS, f"CV must be one of {sorted(FOLDS)}"
assert PRESET in PRESETS, f"PRESET must be one of {sorted(PRESETS)}"

## 3. Helper functions

`load_store_frame` reads each pickle once and keeps only the chosen store's rows and the wanted columns, so memory stays low. If an expected column is missing, it **warns and skips it** (the model then trains on fewer features), so check the warnings in the output.

In [ ]:
def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast integers, turn float64 into float32, and strings into categories."""
    for col in df.columns:
        dtype = df[col].dtype
        if pd.api.types.is_integer_dtype(dtype):
            df[col] = pd.to_numeric(df[col], downcast="integer")
        elif dtype == np.float64:
            df[col] = df[col].astype(np.float32)
        elif dtype == object:
            df[col] = df[col].astype("category")
    return df


def _pick(wanted: list[str], available: pd.Index, name: str) -> list[str]:
    """Keep the wanted columns that exist, and warn loudly about missing ones."""
    present = [c for c in wanted if c in available]
    missing = [c for c in wanted if c not in available]
    if missing:
        LOG.warning("%s: %d expected columns not found, skipping: %s", name, len(missing), missing)
    return present


def _load_subset(path: Path, wanted: list[str], rows: pd.Index, full_index: pd.Index, name: str) -> pd.DataFrame:
    """Read one pickle, keep only the given rows and wanted columns, free the rest."""
    df = pd.read_pickle(path)
    if not df.index.equals(full_index):
        raise ValueError(
            f"{path.name} has a different row index from grid_part_1.pkl, "
            "so the rows cannot be aligned safely."
        )
    subset = df.loc[rows, _pick(wanted, df.columns, name)]
    del df
    gc.collect()
    return subset


def load_store_frame(data_dir: Path, store: str, first_day: int, max_items: int | None) -> pd.DataFrame:
    """Build the feature table for one store."""
    grid1 = pd.read_pickle(data_dir / "grid_part_1.pkl")
    full_index = grid1.index

    keep = (grid1["store_id"] == store) & (grid1["d"] >= first_day)
    if not keep.any():
        raise ValueError(f"No rows found for store {store!r} with d >= {first_day}.")
    base = grid1.loc[keep]
    del grid1
    gc.collect()

    if max_items:
        ids = base["id"].drop_duplicates().iloc[:max_items]
        base = base[base["id"].isin(ids)]
        LOG.info("Using the first %d series only (MAX_ITEMS).", len(ids))

    rows = base.index
    parts = [
        base,
        _load_subset(data_dir / "grid_part_2.pkl", GRID2_COLS, rows, full_index, "price features"),
        _load_subset(data_dir / "grid_part_3.pkl", GRID3_COLS, rows, full_index, "calendar features"),
        _load_subset(data_dir / "lags_df_28.pkl", LAG_COLS, rows, full_index, "lag features"),
        _load_subset(data_dir / "mean_encoding_df.pkl", MEAN_ENC_COLS, rows, full_index, "mean encodings"),
    ]
    df = pd.concat(parts, axis=1)
    del parts, base
    gc.collect()

    df = reduce_mem_usage(df)
    LOG.info("Loaded %s: %d rows x %d columns, %.0f MB",
             store, len(df), df.shape[1], df.memory_usage(deep=True).sum() / 1e6)
    return df


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = y_true.astype(np.float64)
    y_pred = y_pred.astype(np.float64)
    err = y_pred - y_true
    total = float(np.abs(y_true).sum())
    return {
        "rmse": float(np.sqrt(np.mean(err**2))),
        "mae": float(np.mean(np.abs(err))),
        "wape": float(np.abs(err).sum() / total) if total > 0 else float("nan"),
    }


def seasonal_naive(frame: pd.DataFrame, cutoff: int, end: int) -> np.ndarray:
    """Predict each validation day with the same weekday from the last 7 known days.

    `frame` needs columns id, d, sales for days in (cutoff - 7, end]. Only days up
    to `cutoff` are used as history, so there is no peeking into the validation window.
    """
    val = frame.loc[(frame["d"] > cutoff) & (frame["d"] <= end), ["id", "d"]].copy()
    val["id"] = val["id"].astype(str)
    val["d"] = val["d"].astype("int32")
    horizon = val["d"] - cutoff
    val["src"] = val["d"] - 7 * ((horizon + 6) // 7)

    hist = frame.loc[(frame["d"] > cutoff - 7) & (frame["d"] <= cutoff), ["id", "d", TARGET]].copy()
    hist["id"] = hist["id"].astype(str)
    hist["d"] = hist["d"].astype("int32")
    hist = hist.rename(columns={"d": "src", TARGET: "pred"})

    merged = val.merge(hist, on=["id", "src"], how="left")
    return merged["pred"].fillna(0).to_numpy()

## 4. Load the store's data

In [ ]:
cutoff, end = FOLDS[CV]
OUT_DIR.mkdir(parents=True, exist_ok=True)
stem = f"non_recur_{STORE}_{CV}"
model_path = OUT_DIR / f"{stem}.txt"

SKIP = model_path.exists() and not FORCE
if SKIP:
    LOG.info("%s already exists. Set FORCE = True to retrain. The cells below will be skipped.", model_path)
else:
    df = load_store_frame(DATA_DIR, STORE, FIRST_DAY, MAX_ITEMS)
    display(df.head())

## 5. Split by day and compute the baselines

The model never sees the validation days. The baselines are computed here, from a small frame, before the large table is freed.

In [ ]:
if not SKIP:
    day = df["d"]
    train_mask = (day >= FIRST_DAY) & (day <= cutoff) & df[TARGET].notna()
    valid_mask = (day > cutoff) & (day <= end)
    if df.loc[valid_mask, TARGET].isna().any():
        raise ValueError(f"Fold {CV} has missing sales in its validation window, so it cannot be scored.")
    if not train_mask.any() or not valid_mask.any():
        raise ValueError("Empty train or validation split. Check FIRST_DAY and CV.")

    features = [c for c in df.columns if c not in REMOVE_FEATURES]
    LOG.info("Train days %d..%d (%d rows) | validate days %d..%d (%d rows) | %d features",
             FIRST_DAY, cutoff, int(train_mask.sum()), cutoff + 1, end, int(valid_mask.sum()), len(features))

    base_frame = df.loc[(day > cutoff - 7) & (day <= end), ["id", "d", TARGET]]
    y_valid_np = df.loc[valid_mask, TARGET].to_numpy()
    zeros_pred = np.zeros_like(y_valid_np, dtype=np.float64)
    naive_pred = seasonal_naive(base_frame, cutoff, end)

    x_train, y_train = df.loc[train_mask, features], df.loc[train_mask, TARGET]
    x_valid, y_valid = df.loc[valid_mask, features], df.loc[valid_mask, TARGET]
    valid_ids = df.loc[valid_mask, ["id", "d"]].copy()

    # Free the big table. If you want to inspect it, skip this cell's last two lines.
    del df, base_frame
    gc.collect()

## 6. Train

`PRESET = "dev"` is much faster than `"original"`. Time one run first, and use the training time to estimate longer runs.

In [ ]:
if not SKIP:
    preset = PRESETS[PRESET]
    params = {**BASE_PARAMS, **{k: v for k, v in preset.items() if k != "rounds"}}
    if NUM_THREADS:
        params["num_threads"] = NUM_THREADS

    dtrain = lgb.Dataset(x_train, label=y_train)
    dvalid = lgb.Dataset(x_valid, label=y_valid, reference=dtrain)
    callbacks = [lgb.log_evaluation(50)]
    if EARLY_STOPPING > 0:
        callbacks.append(lgb.early_stopping(EARLY_STOPPING, verbose=True))

    start = time.perf_counter()
    model = lgb.train(
        params,
        dtrain,
        num_boost_round=preset["rounds"],
        valid_sets=[dvalid],
        valid_names=["valid"],
        callbacks=callbacks,
    )
    train_seconds = time.perf_counter() - start
    best_iter = model.best_iteration or model.current_iteration()
    LOG.info("Training took %.1f s, best iteration %d", train_seconds, best_iter)

## 7. Evaluate against the baselines

Lower is better for all three numbers. WAPE is the total absolute error divided by total actual sales. If the model does not clearly beat both baselines, it is not ready to use.

In [ ]:
if not SKIP:
    pred = model.predict(x_valid)
    results = {
        "model": regression_metrics(y_valid_np, pred),
        "baseline_zeros": regression_metrics(y_valid_np, zeros_pred),
        "baseline_seasonal_naive": regression_metrics(y_valid_np, naive_pred),
    }
    display(pd.DataFrame(results).T.round(4))

    importance = pd.Series(model.feature_importance("gain"), index=model.feature_name())
    print("Top 15 features by gain")
    display(importance.sort_values(ascending=False).head(15).round(0).to_frame("gain"))

## 8. Save the model, predictions and run details

In [ ]:
if not SKIP:
    model.save_model(str(model_path), num_iteration=best_iter)
    out = valid_ids.assign(y_true=y_valid_np, y_pred=pred, y_seasonal_naive=naive_pred)
    out.to_csv(OUT_DIR / f"{stem}_predictions.csv", index=False)
    meta = {
        "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "store": STORE,
        "fold": CV,
        "train_days": [FIRST_DAY, cutoff],
        "valid_days": [cutoff + 1, end],
        "train_rows": int(len(x_train)),
        "valid_rows": int(len(x_valid)),
        "max_items": MAX_ITEMS,
        "preset": PRESET,
        "params": params,
        "best_iteration": int(best_iter),
        "train_seconds": round(train_seconds, 1),
        "features": features,
        "metrics": results,
        "lightgbm_version": lgb.__version__,
    }
    (OUT_DIR / f"{stem}_meta.json").write_text(json.dumps(meta, indent=2))
    LOG.info("Saved model, predictions and metadata to %s", OUT_DIR)

## Where to go next
- Run the same store with the recursive model (`1-1`) and compare the errors on days 1 to 7 of the window against days 22 to 28.
- Average the two sets of predictions and check whether the error drops. That is a small version of the winning ensemble.
- Add checks that no feature uses data from the validation window.